# Analysis of Ship Wreck Data from the Roman Empire
## Gene Foxwell
## Dataset: https://oxrep.classics.ox.ac.uk/databases/shipwrecks_database/0to20/

Strauss, J. (2013). Shipwrecks Database. Version 1.0. Accessed (date): oxrep.classics.ox.ac.uk/databases/shipwrecks_database/

We will use this dataset to try and determine the most dangerous sailing routes in the Roman Empire. We will take into account the ships origin and destination, the tonnage of the vessel, as well as the sea area. Finally we look at the estimated positions of the wrecks to see if there were specific "danger" zones that put ships in danger more than others.

## Dataset Overview

We will use this dataset to search for patterns in Roman Shipwrecks. We are interested in where they happened, and the characteristics of the ships that they happened to. To this end, we are interested in the following fields:

* **Wreck ID**: Allows us to reference the ship wreck.
* **Sea area**: Tells us what area of the sea the wreck happened in.
* **Lattitude & Longitude**: Rough estimates of the wrecks location. (This were kept impercise in the dataset to discourage looting)
* **Estimated tonnage**: Roughly how large this ship was.
* **Place of origin**: Where the ship was coming from.
* **Place of destination**: Where the ship was going.

## Load and Clean the Dataset

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

In [2]:
df = pd.read_csv("data/StraussShipwrecks.csv")
df.head()

,Wreck ID,Strauss ID,Name,Parker Number,Sea area,Country,Region,Latitude,Longitude,Min depth,...,Columns etc,Sarcophagi,Blocks,Marble type,Other cargo,Hull remains,Shipboard paraphernalia,Ship equipment,Estimated tonnage,Amphora type
0,1,331,Komiza,NaN,Adriatic,Croatia,Vis Island,43.03333,16.08333,30.0,...,False,False,False,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2,328,Lokunji,NaN,Adriatic,Croatia,Kvarner region,44.70000,14.28333,4.0,...,False,False,False,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,3,329,Maharac Cape,NaN,Adriatic,Croatia,Mljet island,42.73333,17.66666,3.0,...,False,False,False,NaN,Eastern coarse ware pottery of biconical dishe...,NaN,NaN,NaN,NaN,NaN
3,4,330,Mlin,702,Adriatic,Croatia,Split channel,43.45000,16.23333,25.0,...,False,False,False,NaN,NaN,Remains of the hull.,NaN,"Two lead anchor stocks, though possibly not fr...",NaN,NaN
4,5,322,Plavac B,832,Adriatic,Croatia,"Zlarin Island, central Dalmatia",NaN,NaN,NaN,...,False,False,False,NaN,NaN,NaN,NaN,.,NaN,NaN


Ok, we have the data. We don't need all those fields. We'll clean this table up a bit so it only has the columns we actually want.

In [3]:
def prepare_shipwreck_data(df_source):
    """
    Drops columns that we won't be using from the dataframe.

    Args:
        df_source: the source dataframe.
    Returns:
        A new dataframe with only the expected columns.
    """    
    df_results = df_source.copy()

    df_results = df_results.drop(columns=["Strauss ID", 
                                          "Country",
                                          "Region",
                                          "Parker Number", 
                                          "Depth", 
                                          "Period", 
                                          "Dating", 
                                          "Earliest date",
                                          "Latest date",
                                          "Date range",
                                          "Mid point of date range", 
                                          "Probability", 
                                          "Reference", 
                                          "Comments", 
                                          "Amphorae", 
                                          "Marble",
                                          "Columns etc",
                                          "Sarcophagi",
                                          "Blocks",
                                          "Marble type",
                                          "Other cargo",
                                          "Hull remains",
                                          "Shipboard paraphernalia",
                                          "Ship equipment",
                                          "Min depth",
                                          "Max depth",
                                          "Amphora type"
                                        ])

    return df_results

In [4]:
df_shipwrecks = prepare_shipwreck_data(df)
df_shipwrecks.head()

,Wreck ID,Name,Sea area,Latitude,Longitude,Place of origin,Place of destination,Estimated tonnage
0,1,Komiza,Adriatic,43.03333,16.08333,Egypt,Northern Italy,NaN
1,2,Lokunji,Adriatic,44.70000,14.28333,Cos,Northern Italy,NaN
2,3,Maharac Cape,Adriatic,42.73333,17.66666,Rhodes,Northern Italy,NaN
3,4,Mlin,Adriatic,43.45000,16.23333,Aegean,Northern Italy,NaN
4,5,Plavac B,Adriatic,NaN,NaN,NaN,NaN,NaN


We can already see some missing values here. We'll handle these on a case by case basis when we analyze the different aspects of the problem. 

Here are the factors we want to investigate regarding Ship Wrecks:

* Frequency of Ship Wrecks by Trade Route. A Trade Route is a "Place of Origin" "Place of Destination" pair.
* Frequency of Ship Wrecks by Sea Area.
* Frequency of Ship Wrecks by Tonnage.
* How does the frequency of Ship Wrecks by Sea Area and Trade Route relate to a Ships tonnage?
* Do any clusters exist in the Lat / Long - areas that are especially dangerous to sail by?

Let's get started.

## **RQ1**: What is the frequency of Ship Wrecks by Trade Route?

First we have to group the ship wrecks into trade routes. To do this, we'll create a new field "Trade Route". Trade Route will be a string that Origination-Destination. We will remove any rows that do not have a recorded point of origin or destination before doing this analysis.

In [ ]:
def create_trade_route_df(df_source):
    """
    Creates a new dataframe that includes a "traderoutes" column. traderoutes is a concatenation of Place of origin and Place of destination. Rows which are missing either a
    Place of origin or a Place of destination are removed before this column is created.

    Args:
        df_source: The source data frame.

    Returns:
        A copy of the source dataframe modified to include the "traderoutes" column as described above.
    """
    df_results = df_source.copy()
    df_results = df_results.drop(columns=["Sea area", "Latitude", "Longitude", "Estimated tonnage"])

    df_results["Place of origin"] = df_results["Place of origin"].replace("?", np.nan, regex=False)
    df_results["Place of destination"] = df_results["Place of destination"].replace("?", np.nan, regex=False)
    
    df_results.dropna(inplace=True)

    df_results["traderoutes"] = df_results["Place of origin"] + "-" + df_results["Place of destination"]
    df_results["traderoutes"] = df_results["traderoutes"].str.replace("?", "", regex=False)

    return df_results

In [16]:
df_traderoutes = create_trade_route_df(df_source=df_shipwrecks)
df_traderoutes.head()

,Wreck ID,Name,Place of origin,Place of destination,traderoutes
0,1,Komiza,Egypt,Northern Italy,Egypt-Northern Italy
1,2,Lokunji,Cos,Northern Italy,Cos-Northern Italy
2,3,Maharac Cape,Rhodes,Northern Italy,Rhodes-Northern Italy
3,4,Mlin,Aegean,Northern Italy,Aegean-Northern Italy
5,6,Tramerka,Rhodes,northern Italy,Rhodes-northern Italy


Now, lets see how many unique traderoutes we can identify from the data. This will inform our approach to charting it.

In [14]:
df_traderoutes["traderoutes"].unique()

array(['Egypt-Northern Italy', 'Cos-Northern Italy',
       'Rhodes-Northern Italy', 'Aegean-Northern Italy',
       'Rhodes-northern Italy', 'Aegean via Cosa-France', 'Aegean.-Rome',
       'Aegean/N. Africa/Propontis-Sicily/Rome',
       'Proconnesus-Claros, Turkey', 'Greece-Northern Italy',
       'Eastern Cilicia-Aegean Rome', 'Cos-Black Sea',
       'Thessaly, Proconnesus-Black Sea', 'Corinth-Italy', 'Aegean-Italy',
       'Corinth-Pergamum', 'Adriatic-Aegean', 'Carystos-Italy',
       'Aegean-Levant', 'Italy-Caesarea', 'Adriatic-Alexandria',
       'Italy via Crete and Rhodes-Alexandria', 'Aegean-North Africa',
       'Thasos-Black Sea', 'Aegean-India', 'Alex/ Pergamum/ Aegean-Rome',
       'Rhodes-Rome', 'Assos-Italy', 'Aegean-Rome', 'Italy-India',
       'Aegean-France/Spain', 'Luna-South of France', 'Cos-Rome',
       'Aegean, Italy-France/Spain', 'Cos, Crete, Italy-France',
       'Luna-Inland France', 'Aegean via southern Italy-France',
       'Egypt Luna-France, Spain', 'Ae